In [1]:
import os
import json
import glob
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoProcessor, LlavaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [2]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head

n_layer = model.config.text_config.num_hidden_layers
hidden_dim = model.config.text_config.hidden_size
print(n_layer, "language model layers,", hidden_dim, "hidden dim")

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

32 language model layers, 4096 hidden dim


In [4]:
BATCH_DIRS = [
    "/kaggle/input/datasets/nocturnalnerd18/hidden-state-cache-llava-batch1",
    "/kaggle/input/datasets/nocturnalnerd18/hidden-state-cache-llava-batch2",
    "/kaggle/input/datasets/nocturnalnerd18/hidden-state-cache-llava-batch3",
    "/kaggle/input/datasets/nocturnalnerd18/hidden-state-cache-llava-batch4",
    "/kaggle/input/datasets/nocturnalnerd18/hidden-state-cache-llava-batch5",
]

def path_to_image_id(path):
    return os.path.splitext(os.path.basename(path))[0]


all_paths = []
for d in BATCH_DIRS:
    all_paths.extend(glob.glob(os.path.join(d, "*.pt")))

print(len(all_paths), "cached image files found")

350 cached image files found


In [6]:
with open("/kaggle/input/datasets/nocturnalnerd18/vlm-splits/pope_split.json") as f:
    pope_ids = set(json.load(f).keys())

filtered_paths = [p for p in all_paths if path_to_image_id(p) not in pope_ids]
print(f"{len(all_paths) - len(filtered_paths)} POPE-overlap images excluded, {len(filtered_paths)} remain")

random.seed(42)
shuffled_paths = filtered_paths.copy()
random.shuffle(shuffled_paths)

split_point = int(len(shuffled_paths) * 0.8)
train_paths = shuffled_paths[:split_point]
val_paths = shuffled_paths[split_point:]

print(f"{len(train_paths)} images for training the translators, {len(val_paths)} held out for evaluating them")

4 POPE-overlap images excluded, 346 remain
276 images for training the translators, 70 held out for evaluating them


In [8]:
def select_training_positions(is_image_at_keep, is_caption_at_keep, mode="all"):
    """Returns kept-position indices to train on."""
    n = is_image_at_keep.shape[0]
    all_positions = torch.arange(n)
    if mode == "all":
        return all_positions
    elif mode == "image":
        return all_positions[is_image_at_keep]
    elif mode == "text":
        return all_positions[is_caption_at_keep]
    else:
        raise ValueError(f"unknown mode: {mode}")


TRAIN_ON = "all"

In [9]:
class TunedLensTranslators(nn.Module):
    def __init__(self, n_layer, hidden_dim):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layer)])
        for layer in self.layers:
            layer.weight.data.copy_(torch.eye(hidden_dim))
            layer.bias.data.zero_()


translators = TunedLensTranslators(n_layer, hidden_dim).to(lm_head.weight.device, torch.float16)
print(sum(p.numel() for p in translators.parameters()), "trainable parameters across", n_layer, "translators")

537001984 trainable parameters across 32 translators


In [10]:
sample_entry = torch.load(train_paths[0])
check_layer = 3
h = sample_entry["hidden_states"][check_layer, -1, :].to(lm_head.weight.device, torch.float16)

plain_logits = lm_head(ln_f(h))
translated_logits = lm_head(ln_f(translators.layers[check_layer](h)))

diff = (plain_logits.float() - translated_logits.float()).abs().max().item()
print("identity-init check, max diff:", diff)
assert diff < 1e-2, "Untrained translator doesn't match plain logit lens -- check identity init."
del sample_entry

identity-init check, max diff: 0.0


In [11]:
def translator_loss(translator, h, target_probs, ln_f, lm_head):
    """KL(target_probs || translated distribution)"""
    translated_logits = lm_head(ln_f(translator(h)))
    translated_log_probs = F.log_softmax(translated_logits.float(), dim=-1)
    return F.kl_div(translated_log_probs, target_probs.float(), reduction="batchmean")

In [ ]:
translator_device = next(translators.parameters()).device
optimizer = torch.optim.SGD(translators.parameters(), lr=0.1, momentum=0.0)
NUM_EPOCHS = 20

loss_curve = []
step = 0
for epoch in range(NUM_EPOCHS):
    for path in train_paths:
        entry = torch.load(path)
        positions = select_training_positions(entry["is_image_at_keep"], entry["is_caption_at_keep"], TRAIN_ON)

        optimizer.zero_grad()
        total_loss_value = 0.0
        for i in range(n_layer):
            h = entry["hidden_states"][i, positions, :].to(translator_device, torch.float16)
            target_probs = entry["target_probs"][positions, :].to(translator_device)
            layer_loss = translator_loss(translators.layers[i], h, target_probs, ln_f, lm_head)
            layer_loss.backward()
            total_loss_value += layer_loss.item()
        del entry

        torch.nn.utils.clip_grad_norm_(translators.parameters(), max_norm=1.0)
        optimizer.step()

        loss_curve.append(total_loss_value)
        if step % 20 == 0:
            print(f"step {step} (epoch {epoch}): total loss {total_loss_value:.4f}")
        step += 1

plt.figure(figsize=(6, 4))
plt.plot(loss_curve)
plt.xlabel("step")
plt.ylabel("summed KL loss across layers")
plt.title(f"Tuned lens training on LLaVA (TRAIN_ON={TRAIN_ON!r}), from cache")
plt.grid(alpha=0.3)
plt.savefig("/kaggle/working/tuned_lens_training_loss.png", dpi=150, bbox_inches="tight")
plt.show()

step 0 (epoch 0): total loss 151.5401
step 20 (epoch 0): total loss 87.6439
step 40 (epoch 0): total loss 85.1475
step 60 (epoch 0): total loss 64.2878
step 80 (epoch 0): total loss 72.5178
step 100 (epoch 0): total loss 64.7066
step 120 (epoch 0): total loss 71.6473
step 140 (epoch 0): total loss 67.9554
step 160 (epoch 0): total loss 61.4402
step 180 (epoch 0): total loss 59.4599
step 200 (epoch 0): total loss 52.0696
step 220 (epoch 0): total loss 46.0556
step 240 (epoch 0): total loss 61.2404
step 260 (epoch 0): total loss 45.8531
step 280 (epoch 1): total loss 49.4080
step 300 (epoch 1): total loss 71.2164
step 320 (epoch 1): total loss 54.0372
step 340 (epoch 1): total loss 44.7652
step 360 (epoch 1): total loss 51.7688
step 380 (epoch 1): total loss 48.3511
step 400 (epoch 1): total loss 42.4245
step 420 (epoch 1): total loss 51.8455


In [ ]:
os.makedirs("/kaggle/working/cache", exist_ok=True)
torch.save(
    {
        "n_layer": n_layer,
        "hidden_dim": hidden_dim,
        "train_on": TRAIN_ON,
        "train_image_ids": [path_to_image_id(p) for p in train_paths],
        "val_image_ids": [path_to_image_id(p) for p in val_paths],
        "translators_state_dict": translators.state_dict(),
        "loss_curve": loss_curve,
        "num_epochs": NUM_EPOCHS,
    },
    "/kaggle/working/cache/tuned_lens_llava.pt",
)
print("saved.")

dataset_id = f"tuned-lens-llava-{TRAIN_ON}"
with open("/kaggle/working/cache/dataset-metadata.json", "w") as f:
    json.dump({
        "title": dataset_id,
        "id": f"nocturnalnerd18/{dataset_id}",
        "licenses": [{"name": "CC0-1.0"}]
    }, f)

!kaggle datasets create -p /kaggle/working/cache/

In [ ]:
@torch.no_grad()
def per_layer_top1_match_rate(paths, translators=None):
    n = n_layer + 1
    match_counts = torch.zeros(n)
    total_counts = torch.zeros(n)

    for path in paths:
        entry = torch.load(path)
        target_top1 = entry["target_probs"].argmax(dim=-1)
        for i in range(n):
            h = entry["hidden_states"][i, :, :].to(lm_head.weight.device, torch.float16)
            if translators is not None and i < n_layer:
                h = translators.layers[i](h)
            if i < n - 1:
                h = ln_f(h)
            logits = lm_head(h)
            lens_top1 = logits.argmax(dim=-1).cpu()
            match_counts[i] += (lens_top1 == target_top1).sum().item()
            total_counts[i] += target_top1.shape[0]
        del entry

    return (match_counts / total_counts).numpy()


plain_rates = per_layer_top1_match_rate(val_paths, translators=None)
tuned_rates = per_layer_top1_match_rate(val_paths, translators=translators)

plt.figure(figsize=(8, 5))
plt.plot(plain_rates, label="plain logit lens")
plt.plot(tuned_rates, label="tuned lens")
plt.xlabel("layer")
plt.ylabel("top-1 match rate vs. model's real output")
plt.title("Tuned lens vs. logit lens, evaluated on held-out images")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("/kaggle/working/tuned_vs_plain_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("plain, by layer:", plain_rates)
print("tuned, by layer:", tuned_rates)